### Evaluación con Voces Reales del Equipo - Modelo 2.1

Probaremos el Modelo 2.1 con los datos balanceados y class weight con audios que grabamos en nuestros telefonos con las voces de integrantes del equipo y otros, para evaluar el performance del modelo con estos audios.

Para esto tuvimos que instalar el convertidor de audio ffmpeg.

In [5]:
import os
import glob
import numpy as np
import librosa
import tensorflow as tf

# ── Parámetros ETL (idénticos al entrenamiento) ──────────────────────────────
N_FRAMES    = 5
FRAME_LEN   = 2048
TARGET_SR   = 16000
RUTA_MODELO = '../Metricas/modelo2_1_balanceado.keras'
RUTA_VOCES  = '../test_voces/'

# ── Cargar modelo ─────────────────────────────────────────────────────────────
modelo = tf.keras.models.load_model(RUTA_MODELO)

# ── Funciones ETL ─────────────────────────────────────────────────────────────
def calcular_segmentos_uniformes(total_samples, n_frames=5, frame_length=2048):
    segment_size = total_samples // n_frames
    indices = []
    for i in range(n_frames):
        center = (i * segment_size) + (segment_size // 2)
        start  = center - (frame_length // 2)
        end    = start + frame_length
        if start < 0:
            start, end = 0, frame_length
        if end > total_samples:
            start, end = total_samples - frame_length, total_samples
        indices.append((start, end))
    return indices

def audio_a_tensor(file_path):
    audio, _ = librosa.load(file_path, sr=TARGET_SR, mono=True)
    if len(audio) < FRAME_LEN:
        raise ValueError(f"Audio demasiado corto: {file_path}")
    frames = calcular_segmentos_uniformes(len(audio), N_FRAMES, FRAME_LEN)
    features = []
    for start, end in frames:
        stft = librosa.stft(audio[start:end], n_fft=FRAME_LEN, hop_length=FRAME_LEN + 1)
        features.append(np.abs(stft))           # (1025, 1)
    tensor = np.array(features)                 # (5, 1025, 1)
    return np.expand_dims(tensor, axis=0)       # (1, 5, 1025, 1)

# ── Seleccionar archivos _16k.wav ─────────────────────────────────────────────
archivos = sorted(glob.glob(os.path.join(RUTA_VOCES, '*_16k.wav')))
print(f"Archivos encontrados: {len(archivos)}\n")

# ── Predecir ──────────────────────────────────────────────────────────────────
resultados = []

for ruta in archivos:
    nombre = os.path.basename(ruta)
    try:
        tensor = audio_a_tensor(ruta)
        prob   = modelo.predict(tensor, verbose=0)[0][0]
        clase  = "SPOOF  ⚠️" if prob > 0.5 else "BONAFIDE ✅"
        resultados.append((nombre, prob, clase))
    except Exception as e:
        resultados.append((nombre, None, f"ERROR: {e}"))

# ── Mostrar tabla de resultados ───────────────────────────────────────────────
print(f"{'Archivo':<45} {'P(spoof)':>10}  {'Resultado'}")
print("─" * 75)
for nombre, prob, clase in resultados:
    prob_str = f"{prob:.4f}" if prob is not None else "  N/A  "
    print(f"{nombre:<45} {prob_str:>10}  {clase}")


Archivos encontrados: 24

Archivo                                         P(spoof)  Resultado
───────────────────────────────────────────────────────────────────────────
AUDIO-2026-04-28-16-00-52_16k.wav                 0.8126  SPOOF  ⚠️
AUDIO-2026-04-28-16-01-01_16k.wav                 0.9999  SPOOF  ⚠️
AlexisLiliana_VozEspañolWhatsiPhone_16k.wav      0.9996  SPOOF  ⚠️
Alexis_VozEspañolMac2_16k.wav                    0.4269  BONAFIDE ✅
Alexis_VozEspañolMac_16k.wav                     0.9403  SPOOF  ⚠️
Alexis_VozEspañolWhatsiPhone_16k.wav             0.9552  SPOOF  ⚠️
Daniele_CancionEspanol_16k.wav                    0.2638  BONAFIDE ✅
Daniele_CancionItaliano_16k.wav                   0.9999  SPOOF  ⚠️
Daniele_VozEspañolMac_16k.wav                    0.4881  BONAFIDE ✅
Daniele_VozInglesMac 2_16k.wav                    0.8987  SPOOF  ⚠️
Daniele_VozInglesMac_16k.wav                      0.8987  SPOOF  ⚠️
Daniele_VozIngleslWhatsiPhone_16k.wav             0.3547  BONAFIDE ✅
Daniele_Vo